In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
# %aimport AAnF_Benchmark_Tool
# AAnF_Benchmark.df_2_breaks

In [31]:
import sys, os
import AAnF_Benchmark_Tool
from AAnF_Benchmark_Tool import AAnF_Benchmark
from AAnF_Benchmark_Tool import *

pd.options.display.float_format = format_float
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows"   , 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x) # Decimal in views

In [32]:
df_raw = pd.read_csv( "query-impala-7310928.csv", sep = ",")
df_raw = df_raw.rename(columns={
    "total_amount": "txn_amt",
    "total_txns": "txn_cnt",
    "appr_amount": "app_amt",
    "appr_txns": "app_cnt",
    "declined_amount": "dcl_amt",
    "declined_txns": "dcl_cnt",
    "qt_fraud": "fraud_cnt",
    "fraud_amount": "fraud_amt"
})

df_raw['fraud_cnt'] = 0
df_raw['fraud_amt'] = 0
    
df_raw.describe()

,txn_amt,app_cnt,app_amt,dcl_cnt,dcl_amt,amount_txn_pure,txn_cnt,fraud_cnt,fraud_amt
count,1789.00,1789.00,1789.00,1789.00,1789.00,1789.00,1789.00,1789.00,1789.00
mean,16071639.61,13576.29,6628402.61,6415.40,9443237.00,16071639.61,19991.69,0.00,0.00
std,257127413.07,47039.11,23251157.72,23203.52,252975613.76,257127413.07,61995.40,0.00,0.00
min,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
25%,35204.79,33.00,16511.07,22.00,8760.93,35204.79,65.00,0.00,0.00
50%,448613.93,484.00,269857.37,218.00,150501.80,448613.93,842.00,0.00,0.00
75%,4564505.33,5111.00,2562889.44,3324.00,1694866.40,4564505.33,9149.00,0.00,0.00
max,10803904996.84,463546.00,286578655.30,282667.00,10694734709.86,10803904996.84,475831.00,0.00,0.00


In [33]:
industries = [
    "Restaurants",
    "Accommodations",
    "Groceries",
    "General Retail",
    "Fuel & Auto",
    "Finance and Insurance ",
    "Transport & Warehousing",
    "Consumer Electronics",
    "Entertainment & Recreation",
    "Apparel",
    "Other Services",
    "Public Administration",
    "Telco & Network ",
    "Travel Agencies & Remediation Services",
    "Wholesale Trade",
    "Professional Services",
    "Sports & Hobby",
    "Miscellaneous",
    "Education Services",
    "Construction Services"
]

selected_industries = ["Others", "Miscellaneous", "Consumer Electronics", "Travel Agencies & Remediation Services", "Accommodations", "Finance and Insurance ", "Telco & Network "]
df_raw.loc[~df_raw["super_industry_name_grouped"].isin(industries), "super_industry_name_grouped"] = "Others"
df_raw.loc[~df_raw["super_industry_name_grouped"].isin(selected_industries), "super_industry_name_grouped"] = "DISCARD"
df_raw.loc[df_raw["super_industry_name_grouped"].isin(["Travel Agencies & Remediation Services", "Accommodations"]), "super_industry_name_grouped"] = "T&E"
df_raw["super_industry_name_grouped"].value_counts(dropna=False)

super_industry_name_grouped
DISCARD                   897
Others                    503
T&E                       126
Telco & Network            68
Miscellaneous              68
Consumer Electronics       65
Finance and Insurance      62
Name: count, dtype: int64

In [ ]:
# df_raw = df_raw[df_raw["wallet_flag"] == "WALLET"]
# df_raw = df_raw[df_raw["super_industry_name_grouped"] != "DISCARD"]

In [35]:
rows = ["issuer_name"]
break_cols = rows + ["super_industry_name_grouped"] 
columns_pivot = ["app","txn","fraud"]
columns_to_sum = [ col + "_cnt" for col in columns_pivot]
columns_to_sum

['app_cnt', 'txn_cnt', 'fraud_cnt']

In [36]:
df_pivot = df_raw.groupby( break_cols ) \
    .agg( { col: "sum" for col in columns_to_sum  }  )  \
    .reset_index() 
df_pivot

,issuer_name,super_industry_name_grouped,app_cnt,txn_cnt,fraud_cnt
0,BANCO BRADESCO S.A.,Others,1,1,0
1,BANCO BTG PACTUAL SA,Consumer Electronics,0,4,0
2,BANCO BTG PACTUAL SA,Finance and Insurance,0,1,0
3,BANCO BTG PACTUAL SA,Miscellaneous,9040,9302,0
4,BANCO BTG PACTUAL SA,Others,0,1,0
5,BANCO BTG PACTUAL SA,T&E,0,2,0
6,BANCO BTG PACTUAL SA,Telco & Network,2,18,0
7,BANCO C6 SA,Consumer Electronics,996,2879,0
8,BANCO C6 SA,Finance and Insurance,71,306,0
9,BANCO C6 SA,Miscellaneous,2214,4651,0


In [37]:
break_cols = [item for item in break_cols if item not in rows]
df_pivot_adj = pd \
    .pivot_table(df_pivot, 
                 index = rows,
           columns = break_cols  ,
           values = columns_to_sum, 
           aggfunc = "sum"
           ) \
    .reset_index()

df_pivot_adj

issuer_name  \
super_industry_name_grouped                                  
0                                      BANCO BRADESCO S.A.   
1                                     BANCO BTG PACTUAL SA   
2                                              BANCO C6 SA   
3                                         BANCO INTER S.A.   
4                            BANCO SANTANDER (BRASIL) S.A.   
5                                       ITAU UNIBANCO S.A.   

                                         app_cnt                         \
super_industry_name_grouped Consumer Electronics Finance and Insurance    
0                                            NaN                    NaN   
1                                           0.00                   0.00   
2                                         996.00                  71.00   
3                                         270.00                  49.00   
4                                        1124.00                 116.00   
5                                          96.00                   1.00   

                                                                          \
super_industry_name_grouped Miscellaneous Others    T&E Telco & Network    
0                                     NaN   1.00    NaN              NaN   
1                                 9040.00   0.00   0.00             2.00   
2                                 2214.00 339.00 371.00          2646.00   
3                               382482.00 236.00  58.00          1932.00   
4                                 1696.00 411.00 500.00          2851.00   
5                                57176.00  46.00  18.00            52.00   

                                       fraud_cnt                         \
super_industry_name_grouped Consumer Electronics Finance and Insurance    
0                                            NaN                    NaN   
1                                           0.00                   0.00   
2                                           0.00                   0.00   
3                                           0.00                   0.00   
4                                           0.00                   0.00   
5                                           0.00                   0.00   

                                                                        \
super_industry_name_grouped Miscellaneous Others  T&E Telco & Network    
0                                     NaN   0.00  NaN              NaN   
1                                    0.00   0.00 0.00             0.00   
2                                    0.00   0.00 0.00             0.00   
3                                    0.00   0.00 0.00             0.00   
4                                    0.00   0.00 0.00             0.00   
5                                    0.00   0.00 0.00             0.00   

                                         txn_cnt                         \
super_industry_name_grouped Consumer Electronics Finance and Insurance    
0                                            NaN                    NaN   
1                                           4.00                   1.00   
2                                        2879.00                 306.00   
3                                        1009.00                 196.00   
4                                        2048.00                 287.00   
5                                         111.00                   1.00   

                                                                           
super_industry_name_grouped Miscellaneous  Others    T&E Telco & Network   
0                                     NaN    1.00    NaN              NaN  
1                                 9302.00    1.00   2.00            18.00  
2                                 4651.00 1932.00 962.00          5555.00  
3                               409674.00 1294.00 228.00          3536.00  
4                                 2578.00 1154.00 796.00          5169.00  
5            

In [38]:
df_pivot_adj = df_pivot_adj.fillna(0)
df_pivot_adj

issuer_name  \
super_industry_name_grouped                                  
0                                      BANCO BRADESCO S.A.   
1                                     BANCO BTG PACTUAL SA   
2                                              BANCO C6 SA   
3                                         BANCO INTER S.A.   
4                            BANCO SANTANDER (BRASIL) S.A.   
5                                       ITAU UNIBANCO S.A.   

                                         app_cnt                         \
super_industry_name_grouped Consumer Electronics Finance and Insurance    
0                                           0.00                   0.00   
1                                           0.00                   0.00   
2                                         996.00                  71.00   
3                                         270.00                  49.00   
4                                        1124.00                 116.00   
5                                          96.00                   1.00   

                                                                          \
super_industry_name_grouped Miscellaneous Others    T&E Telco & Network    
0                                    0.00   1.00   0.00             0.00   
1                                 9040.00   0.00   0.00             2.00   
2                                 2214.00 339.00 371.00          2646.00   
3                               382482.00 236.00  58.00          1932.00   
4                                 1696.00 411.00 500.00          2851.00   
5                                57176.00  46.00  18.00            52.00   

                                       fraud_cnt                         \
super_industry_name_grouped Consumer Electronics Finance and Insurance    
0                                           0.00                   0.00   
1                                           0.00                   0.00   
2                                           0.00                   0.00   
3                                           0.00                   0.00   
4                                           0.00                   0.00   
5                                           0.00                   0.00   

                                                                        \
super_industry_name_grouped Miscellaneous Others  T&E Telco & Network    
0                                    0.00   0.00 0.00             0.00   
1                                    0.00   0.00 0.00             0.00   
2                                    0.00   0.00 0.00             0.00   
3                                    0.00   0.00 0.00             0.00   
4                                    0.00   0.00 0.00             0.00   
5                                    0.00   0.00 0.00             0.00   

                                         txn_cnt                         \
super_industry_name_grouped Consumer Electronics Finance and Insurance    
0                                           0.00                   0.00   
1                                           4.00                   1.00   
2                                        2879.00                 306.00   
3                                        1009.00                 196.00   
4                                        2048.00                 287.00   
5                                         111.00                   1.00   

                                                                           
super_industry_name_grouped Miscellaneous  Others    T&E Telco & Network   
0                                    0.00    1.00   0.00             0.00  
1                                 9302.00    1.00   2.00            18.00  
2                                 4651.00 1932.00 962.00          5555.00  
3                               409674.00 1294.00 228.00          3536.00  
4                                 2578.00 1154.00 796.00          5169.00  
5            

In [39]:
row_totals = df_pivot_adj.groupby(level=0, axis=1).sum()
row_totals.columns = [ col if col in rows else "General_" + col for col in row_totals.columns]
row_totals

C:\Users\e176097\AppData\Local\Temp\1\ipykernel_37500\1690756666.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  row_totals = df_pivot_adj.groupby(level=0, axis=1).sum()


,General_app_cnt,General_fraud_cnt,issuer_name,General_txn_cnt
0,1.00,0.00,BANCO BRADESCO S.A.,1.00
1,9042.00,0.00,BANCO BTG PACTUAL SA,9328.00
2,6637.00,0.00,BANCO C6 SA,16285.00
3,385027.00,0.00,BANCO INTER S.A.,415937.00
4,6698.00,0.00,BANCO SANTANDER (BRASIL) S.A.,12032.00
5,57389.00,0.00,ITAU UNIBANCO S.A.,71315.00


In [40]:
df_pivot_adj.columns = df_pivot_adj.columns \
    .to_flat_index() \
    .map( lambda x: x[1:] + ( x[0], ) ) \
    .map( "_".join) \
    .str.lstrip( "_")
df_pivot_adj

,issuer_name,Consumer Electronics_app_cnt,Finance and Insurance _app_cnt,Miscellaneous_app_cnt,Others_app_cnt,T&E_app_cnt,Telco & Network _app_cnt,Consumer Electronics_fraud_cnt,Finance and Insurance _fraud_cnt,Miscellaneous_fraud_cnt,Others_fraud_cnt,T&E_fraud_cnt,Telco & Network _fraud_cnt,Consumer Electronics_txn_cnt,Finance and Insurance _txn_cnt,Miscellaneous_txn_cnt,Others_txn_cnt,T&E_txn_cnt,Telco & Network _txn_cnt
0,BANCO BRADESCO S.A.,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
1,BANCO BTG PACTUAL SA,0.00,0.00,9040.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,1.00,9302.00,1.00,2.00,18.00
2,BANCO C6 SA,996.00,71.00,2214.00,339.00,371.00,2646.00,0.00,0.00,0.00,0.00,0.00,0.00,2879.00,306.00,4651.00,1932.00,962.00,5555.00
3,BANCO INTER S.A.,270.00,49.00,382482.00,236.00,58.00,1932.00,0.00,0.00,0.00,0.00,0.00,0.00,1009.00,196.00,409674.00,1294.00,228.00,3536.00
4,BANCO SANTANDER (BRASIL) S.A.,1124.00,116.00,1696.00,411.00,500.00,2851.00,0.00,0.00,0.00,0.00,0.00,0.00,2048.00,287.00,2578.00,1154.00,796.00,5169.00
5,ITAU UNIBANCO S.A.,96.00,1.00,57176.00,46.00,18.00,52.00,0.00,0.00,0.00,0.00,0.00,0.00,111.00,1.00,71042.00,74.00,19.00,68.00


In [41]:
N = df_pivot_adj.shape
df_pivot_adj = df_pivot_adj.merge( row_totals , on = rows )
print( N, row_totals.shape , df_pivot_adj.shape )

(6, 19) (6, 4) (6, 22)


In [42]:
df_pivot_adj

,issuer_name,Consumer Electronics_app_cnt,Finance and Insurance _app_cnt,Miscellaneous_app_cnt,Others_app_cnt,T&E_app_cnt,Telco & Network _app_cnt,Consumer Electronics_fraud_cnt,Finance and Insurance _fraud_cnt,Miscellaneous_fraud_cnt,Others_fraud_cnt,T&E_fraud_cnt,Telco & Network _fraud_cnt,Consumer Electronics_txn_cnt,Finance and Insurance _txn_cnt,Miscellaneous_txn_cnt,Others_txn_cnt,T&E_txn_cnt,Telco & Network _txn_cnt,General_app_cnt,General_fraud_cnt,General_txn_cnt
0,BANCO BRADESCO S.A.,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00
1,BANCO BTG PACTUAL SA,0.00,0.00,9040.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,1.00,9302.00,1.00,2.00,18.00,9042.00,0.00,9328.00
2,BANCO C6 SA,996.00,71.00,2214.00,339.00,371.00,2646.00,0.00,0.00,0.00,0.00,0.00,0.00,2879.00,306.00,4651.00,1932.00,962.00,5555.00,6637.00,0.00,16285.00
3,BANCO INTER S.A.,270.00,49.00,382482.00,236.00,58.00,1932.00,0.00,0.00,0.00,0.00,0.00,0.00,1009.00,196.00,409674.00,1294.00,228.00,3536.00,385027.00,0.00,415937.00
4,BANCO SANTANDER (BRASIL) S.A.,1124.00,116.00,1696.00,411.00,500.00,2851.00,0.00,0.00,0.00,0.00,0.00,0.00,2048.00,287.00,2578.00,1154.00,796.00,5169.00,6698.00,0.00,12032.00
5,ITAU UNIBANCO S.A.,96.00,1.00,57176.00,46.00,18.00,52.00,0.00,0.00,0.00,0.00,0.00,0.00,111.00,1.00,71042.00,74.00,19.00,68.00,57389.00,0.00,71315.00


In [43]:
dict2convert = {
    "Approved" : "app" , 
    "Total"    : "txn", 
    "Fraud"    : "fraud" 
}

In [44]:
breaks_unfiltered = list(df_pivot_adj.columns)[1:]
test = breaks_unfiltered[0]
"".join(test)
breaks = ["General"] + ["_".join(break_var.split("_")[:len(break_cols)]) for break_var in breaks_unfiltered[:len(breaks_unfiltered) - 3]]
breaks = list(dict.fromkeys(breaks))

In [45]:
bank_name = "BANCO SANTANDER (BRASIL) S.A."
iss_col = "issuer_name"

benchmark_tool_Cnt = \
    AAnF_Benchmark(iss_name     = bank_name ,
                   iss_var_name = iss_col,
                   cnt_amt_flag = "Cnt", 
                   breaks       = breaks
                  )

print( benchmark_tool_Cnt.display_attributes() ) 

benchmark_tool_Cnt.df_2_input(df_pivot_adj ,
                              dict2convert ,
                              agg_by = iss_col
                              )

benchmark_tool_Cnt.df_input

benchmark_tool_Cnt.df_2_breaks(dummy_fraud = False)
benchmark_tool_Cnt.get_peer_distances( dict_compare = { "General_Cnt_tl0" : ["Perc_CNP_Cnt"] } ) 
benchmark_tool_Cnt.get_initial_df()

print(benchmark_tool_Cnt.df_input.shape , "\n" ,
      benchmark_tool_Cnt.df_input_fraud.shape , "\n" ,
      benchmark_tool_Cnt.df_breaks.shape , "\n" ,
      benchmark_tool_Cnt.initial_df.shape , "\n" ,
      benchmark_tool_Cnt.breaks
     )

[ ( i, c )  for i,c in enumerate( benchmark_tool_Cnt.initial_df.columns ) if c.startswith( "v_") ]

iss_name     : BANCO SANTANDER (BRASIL) S.A.
iss_var_name : issuer_name
breaks       : ['General', 'Consumer Electronics', 'Finance and Insurance ', 'Miscellaneous', 'Others', 'T&E', 'Telco & Network ']
cnt_amt_flag : Cnt
None
 >> DF input values in df_input
 >> Breaks calculated in df_breaks
 >> Distance with peers in peer_distances
(6, 22) 
 (0, 0) 
 (6, 29) 
 (5, 30) 
 ['General', 'Consumer Electronics', 'Finance and Insurance ', 'Miscellaneous', 'Others', 'T&E', 'Telco & Network ']


[(1, 'v_General_Cnt_br0'),
 (5, 'v_Consumer Electronics_Cnt_br1'),
 (9, 'v_Finance and Insurance _Cnt_br2'),
 (13, 'v_Miscellaneous_Cnt_br3'),
 (17, 'v_Others_Cnt_br4'),
 (21, 'v_T&E_Cnt_br5'),
 (25, 'v_Telco & Network _Cnt_br6')]

In [46]:
benchmark_tool_Cnt.initial_df

,Issuer Name,v_General_Cnt_br0,General_Cnt_tl0,General_Cnt_ap0,General_Cnt_fr0,v_Consumer Electronics_Cnt_br1,Consumer Electronics_Cnt_tl1,Consumer Electronics_Cnt_ap1,Consumer Electronics_Cnt_fr1,v_Finance and Insurance _Cnt_br2,Finance and Insurance _Cnt_tl2,Finance and Insurance _Cnt_ap2,Finance and Insurance _Cnt_fr2,v_Miscellaneous_Cnt_br3,Miscellaneous_Cnt_tl3,Miscellaneous_Cnt_ap3,Miscellaneous_Cnt_fr3,v_Others_Cnt_br4,Others_Cnt_tl4,Others_Cnt_ap4,Others_Cnt_fr4,v_T&E_Cnt_br5,T&E_Cnt_tl5,T&E_Cnt_ap5,T&E_Cnt_fr5,v_Telco & Network _Cnt_br6,Telco & Network _Cnt_tl6,Telco & Network _Cnt_ap6,Telco & Network _Cnt_fr6,distance
0,BANCO BRADESCO S.A.,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,BANCO BTG PACTUAL SA,9042.00,9328.00,9042.00,0.00,0.00,4.00,0.00,0.00,0.00,1.00,0.00,0.00,9040.00,9302.00,9040.00,0.00,0.00,1.00,0.00,0.00,0.00,2.00,0.00,0.00,2.00,18.00,2.00,0.00,0.00
2,BANCO C6 SA,6637.00,16285.00,6637.00,0.00,996.00,2879.00,996.00,0.00,71.00,306.00,71.00,0.00,2214.00,4651.00,2214.00,0.00,339.00,1932.00,339.00,0.00,371.00,962.00,371.00,0.00,2646.00,5555.00,2646.00,0.00,0.00
3,BANCO INTER S.A.,385027.00,415937.00,385027.00,0.00,270.00,1009.00,270.00,0.00,49.00,196.00,49.00,0.00,382482.00,409674.00,382482.00,0.00,236.00,1294.00,236.00,0.00,58.00,228.00,58.00,0.00,1932.00,3536.00,1932.00,0.00,0.00
5,ITAU UNIBANCO S.A.,57389.00,71315.00,57389.00,0.00,96.00,111.00,96.00,0.00,1.00,1.00,1.00,0.00,57176.00,71042.00,57176.00,0.00,46.00,74.00,46.00,0.00,18.00,19.00,18.00,0.00,52.00,68.00,52.00,0.00,0.00


In [47]:
#Só Amount
# benchmark_tool_Amt = \
#     AAnF_Benchmark(iss_name     = bank_name ,
#                    iss_var_name = iss_col,
#                    cnt_amt_flag = "Amt", 
#                    breaks       = breaks
#                   )

# print( benchmark_tool_Amt.display_attributes() ) 

# benchmark_tool_Amt.df_2_input(df_pivot_adj ,
#                               dict2convert ,
#                               agg_by = iss_col
#                               )

# benchmark_tool_Amt.df_2_breaks(dummy_fraud = False)

# benchmark_tool_Amt.get_peer_distances( dict_compare = { "General_Amt_tl0" : ["Perc_CNP_Amt"] } ) 
# benchmark_tool_Amt.get_initial_df()

# print(benchmark_tool_Amt.df_input.shape , "\n" ,
#       benchmark_tool_Amt.df_input_fraud.shape , "\n" ,
#       benchmark_tool_Amt.df_breaks.shape , "\n" ,
#       benchmark_tool_Amt.initial_df.shape , "\n" ,
#       benchmark_tool_Amt.breaks
#      )

# [ ( i, c )  for i,c in enumerate( benchmark_tool_Amt.initial_df.columns ) if c.startswith( "v_") ]

In [48]:
initial_df = benchmark_tool_Cnt.initial_df 
# ini_df_amt = benchmark_tool_Amt.initial_df.drop( columns = ["Issuer Name", 'distance'] )

# patterns = r'_br|_tl|_ap|_fr'
# regex = re.compile(r'({0})(.+)'.format(patterns))
# last_idx = int( regex.search( benchmark_tool_Cnt.df_breaks.columns[ -1: ][0] )[2] ) + 1
# new_names = {}
# for name in ini_df_amt.columns : 
#     match = regex.search(name)
#     new_names[ name ] = re.sub( match[2] , str( int( match.group(2) ) + last_idx ) , name ) 
#     #print( name, " ", match[1] )
#     #print( name , " ", match[2]  , re.sub( match[2] , str( int( match.group(2) ) + last_idx ) , name ) )

# ini_df_amt.rename( columns = new_names, inplace = True)

# initial_df = initial_df.join( ini_df_amt )
initial_df.columns = initial_df.columns.str.strip()

print( initial_df.shape )
[ ( i,c)  for i,c in enumerate(initial_df.columns) if re.search("^[v_|Iss]", c, re.IGNORECASE)  ]

(5, 30)


[(0, 'Issuer Name'),
 (1, 'v_General_Cnt_br0'),
 (5, 'v_Consumer Electronics_Cnt_br1'),
 (9, 'v_Finance and Insurance _Cnt_br2'),
 (13, 'v_Miscellaneous_Cnt_br3'),
 (17, 'v_Others_Cnt_br4'),
 (21, 'v_T&E_Cnt_br5'),
 (25, 'v_Telco & Network _Cnt_br6')]

In [49]:
# Select only the columns that start with 'v_' and the 'Issuer Name' column. 
# Please note that the Issuer Name column is a mandatory column. The name has to be as it is.
selected_columns = ['Issuer Name'] + [col for col in initial_df.columns if col.startswith('v_')]
# Select only the dynamic breaks variables name without Issue Name for iterate later.
selected_columns_num_v = [col for col in initial_df.columns if col.startswith('v_')]
if 'Distance to Client' in initial_df.columns:
    selected_columns_num_v_sort = ['Distance to Client'] + [col for col in initial_df.columns if col.startswith('v_')]
    sort_conditions = [True] + [False for col in initial_df.columns if col.startswith('v_')]
else:
    selected_columns_num_v_sort = [col for col in initial_df.columns if col.startswith('v_')]
    sort_conditions = [False for col in initial_df.columns if col.startswith('v_')]

selected_columns_num_v_sort

['v_General_Cnt_br0',
 'v_Consumer Electronics_Cnt_br1',
 'v_Finance and Insurance _Cnt_br2',
 'v_Miscellaneous_Cnt_br3',
 'v_Others_Cnt_br4',
 'v_T&E_Cnt_br5',
 'v_Telco & Network _Cnt_br6']

In [50]:
initial_df

,Issuer Name,v_General_Cnt_br0,General_Cnt_tl0,General_Cnt_ap0,General_Cnt_fr0,v_Consumer Electronics_Cnt_br1,Consumer Electronics_Cnt_tl1,Consumer Electronics_Cnt_ap1,Consumer Electronics_Cnt_fr1,v_Finance and Insurance _Cnt_br2,Finance and Insurance _Cnt_tl2,Finance and Insurance _Cnt_ap2,Finance and Insurance _Cnt_fr2,v_Miscellaneous_Cnt_br3,Miscellaneous_Cnt_tl3,Miscellaneous_Cnt_ap3,Miscellaneous_Cnt_fr3,v_Others_Cnt_br4,Others_Cnt_tl4,Others_Cnt_ap4,Others_Cnt_fr4,v_T&E_Cnt_br5,T&E_Cnt_tl5,T&E_Cnt_ap5,T&E_Cnt_fr5,v_Telco & Network _Cnt_br6,Telco & Network _Cnt_tl6,Telco & Network _Cnt_ap6,Telco & Network _Cnt_fr6,distance
0,BANCO BRADESCO S.A.,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,BANCO BTG PACTUAL SA,9042.00,9328.00,9042.00,0.00,0.00,4.00,0.00,0.00,0.00,1.00,0.00,0.00,9040.00,9302.00,9040.00,0.00,0.00,1.00,0.00,0.00,0.00,2.00,0.00,0.00,2.00,18.00,2.00,0.00,0.00
2,BANCO C6 SA,6637.00,16285.00,6637.00,0.00,996.00,2879.00,996.00,0.00,71.00,306.00,71.00,0.00,2214.00,4651.00,2214.00,0.00,339.00,1932.00,339.00,0.00,371.00,962.00,371.00,0.00,2646.00,5555.00,2646.00,0.00,0.00
3,BANCO INTER S.A.,385027.00,415937.00,385027.00,0.00,270.00,1009.00,270.00,0.00,49.00,196.00,49.00,0.00,382482.00,409674.00,382482.00,0.00,236.00,1294.00,236.00,0.00,58.00,228.00,58.00,0.00,1932.00,3536.00,1932.00,0.00,0.00
5,ITAU UNIBANCO S.A.,57389.00,71315.00,57389.00,0.00,96.00,111.00,96.00,0.00,1.00,1.00,1.00,0.00,57176.00,71042.00,57176.00,0.00,46.00,74.00,46.00,0.00,18.00,19.00,18.00,0.00,52.00,68.00,52.00,0.00,0.00


In [51]:
# Execute this line to get the combinations
# Defining the benckmark rules. Each are set +1 
rules = [(5,26), (6,31), (7,36), (10,41) ]
# Assigning the breaks variables that will be evaluated in the process
vars_to_eval = selected_columns_num_v_sort # ['v_CNP_DM_CR_APRVOL_br4'] 
## Select the benchmark rule you want to evaluate 
num_participants = 4
max_percentage = 35  # Max percentage each total amount should not exceed. Rule max
## Select the amount of distinct combinations of suggested peers you want to see.
num_combinations = 5

print( " >>", "\n >> ".join( selected_columns_num_v_sort ) , "\n" )

start_time = time.time()
tuple_final = get_combinations_proposed(initial_df, 
                                        num_participants, 
                                        max_percentage,
                                        num_combinations,
                                        vars_to_eval , 
                                        rules)

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.4f} seconds")

 >> v_General_Cnt_br0
 >> v_Consumer Electronics_Cnt_br1
 >> v_Finance and Insurance _Cnt_br2
 >> v_Miscellaneous_Cnt_br3
 >> v_Others_Cnt_br4
 >> v_T&E_Cnt_br5
 >> v_Telco & Network _Cnt_br6 

EVALUATING v_General_Cnt_br0:
Unable to find more combinations.
No combinations under rule choose. Weighting: 
1
3
3
3
3
4

Validated Data:
Combination 1:
            Issuer Name  Distance to Client  v_General_Cnt_br0  \
1  BANCO BTG PACTUAL SA                   0            7052.87   
3      BANCO INTER S.A.                   0            7052.87   
2           BANCO C6 SA                   0            6637.00   
0   BANCO BRADESCO S.A.                   0               1.00   

   v_General_Cnt_br0_Percentage  Factor_Used  
1                         34.00         0.78  
3                         34.00         0.02  
2                         32.00         1.00  
0                          0.00         1.00  

Combination 2:
            Issuer Name  Distance to Client  v_General_Cnt_br0  \
1  

c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\new_flag_wallet\AAnF_Benchmark_Tool.py:628: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  values.loc[i ] = max_allowed # Update the value to the maximum allowed value
c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\new_flag_wallet\AAnF_Benchmark_Tool.py:628: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  values.loc[i ] = max_allowed # Update the value to the maximum allowed value
c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\new_flag_wallet\AAnF_Benchmark_Tool.py:628: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

Unable to find more combinations.
No combinations under rule choose. Weighting: 
No combinations after rule weighting. Evaluating other rules: 
EVALUATING v_Finance and Insurance _Cnt_br2 under rule (5, 26)
Unable to find more combinations.
EVALUATING v_Finance and Insurance _Cnt_br2 under rule (6, 31)
Unable to find more combinations.
EVALUATING v_Finance and Insurance _Cnt_br2 under rule (7, 36)
Unable to find more combinations.
EVALUATING v_Finance and Insurance _Cnt_br2 under rule (10, 41)
Unable to find more combinations.
SORRY, no combinations found under any rule :(
EVALUATING v_Miscellaneous_Cnt_br3:
Unable to find more combinations.
No combinations under rule choose. Weighting: 
1
4

Validated Data:
Combination 1:
            Issuer Name  Distance to Client  v_Miscellaneous_Cnt_br3  \
3    ITAU UNIBANCO S.A.                   0                 11957.37   
2      BANCO INTER S.A.                   0                 11957.37   
0  BANCO BTG PACTUAL SA                   0        

c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\new_flag_wallet\AAnF_Benchmark_Tool.py:628: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  values.loc[i ] = max_allowed # Update the value to the maximum allowed value
c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\new_flag_wallet\AAnF_Benchmark_Tool.py:628: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  values.loc[i ] = max_allowed # Update the value to the maximum allowed value
c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\new_flag_wallet\AAnF_Benchmark_Tool.py:628: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

4

Validated Data:
Combination 1:
            Issuer Name  Distance to Client  v_Telco & Network _Cnt_br6  \
1           BANCO C6 SA                   0                       57.37   
2      BANCO INTER S.A.                   0                       57.37   
3    ITAU UNIBANCO S.A.                   0                       52.00   
0  BANCO BTG PACTUAL SA                   0                        2.00   

   v_Telco & Network _Cnt_br6_Percentage  Factor_Used  
1                                  34.00         0.02  
2                                  34.00         0.03  
3                                  30.81         1.00  
0                                   1.19         1.00  

Combination 2:
            Issuer Name  Distance to Client  v_Telco & Network _Cnt_br6  \
0      BANCO INTER S.A.                   0                       63.09   
2           BANCO C6 SA                   0                       63.09   
3           BANCO C6 SA                   0                       57.

In [52]:
weighted_breaks_list = []
df_kpis = pd.DataFrame( columns=['Metric', 
                                 'AVG Aproval Rate', 
                                 'BIC Approval Rate',
                                 'AVG BPS', 
                                 'BIC BPS',
                                ])

for var in vars_to_eval:
    try:
        selected_comb = 1#input('Select Combination Number for ' + var + ':')
        selected_comb = int(selected_comb)
    except ValueError:
        print("Error: Please enter a valid integer.")

    result_break = get_comb_kpis(var, tuple_final ,selected_comb, bic_apr_q = 0.85 )
    if not result_break:
        continue
    series_to_append = pd.Series(result_break, index=df_kpis.columns)
    df_kpis = pd.concat([df_kpis, series_to_append.to_frame().T], ignore_index=True)

Name:v_General_Cnt_br0
            Issuer Name  General_Cnt_tl0  General_Cnt_ap0  General_Cnt_fr0  \
0   BANCO BRADESCO S.A.             1.00             1.00             0.00   
1  BANCO BTG PACTUAL SA          7275.96          7052.87             0.00   
2           BANCO C6 SA         16285.00          6637.00             0.00   
3      BANCO INTER S.A.          7619.08          7052.87             0.00   

   Factor_Used  apr_rate  frd_bps  
0         1.00      1.00     0.00  
1         0.78      0.97     0.00  
2         1.00      0.41     0.00  
3         0.02      0.93     0.00  
Name:v_General_Cnt_br0
            Issuer Name  General_Cnt_tl0  General_Cnt_ap0  General_Cnt_fr0  \
0   BANCO BRADESCO S.A.             1.00             1.00             0.00   
1  BANCO BTG PACTUAL SA          7275.96          7052.87             0.00   
2           BANCO C6 SA         16285.00          6637.00             0.00   
3      BANCO INTER S.A.          7619.08          7052.87             0

In [53]:
df_kpis

,Metric,AVG Aproval Rate,BIC Approval Rate,AVG BPS,BIC BPS
0,v_General_Cnt_br0,0.67,0.99,0.00,0.00
1,v_Miscellaneous_Cnt_br3,0.85,0.95,0.00,0.00
2,v_Others_Cnt_br4,0.23,0.83,0.00,0.00
3,v_Telco & Network _Cnt_br6,0.54,0.67,0.00,0.00


In [54]:
for dfs in df_kpis: 
    print(dfs)

Metric
AVG Aproval Rate
BIC Approval Rate
AVG BPS
BIC BPS


In [55]:
# with pd.ExcelWriter('output_monthly_vision.xlsx') as writer:
#     # Escribe cada DataFrame filtrado
#     # Verifica si las columnas existen antes de filtrar
#     dfs.to_excel(writer, sheet_name='Sheet1', index=False, startrow=start_row)
#     start_row += len(dfs) + 2  # Espacio después del DataFrame
#     # Espacio extra entre grupos
#     start_row += 1

In [ ]:
df_kpis.to_excel('santander_benchmark_total_monthly_wallet_vision_authO_KPIs.xlsx', index=False)

: 